# Spatial Anonymization – Usage Examples

This notebook demonstrates practical ways to use the **spatial anonymization** library
to anonymize survey GPS coordinates while preserving analytical utility.

---

## 1. Spatial Anonymization: Overview and Concepts

### 1.1 What Is Spatial Anonymization?

Spatial anonymization protects respondent privacy by deliberately introducing
uncertainty into GPS coordinates, while preserving geographic patterns that remain
useful for analysis.

This library implements three commonly used anonymization approaches:

- **DHS-style displacement**  
  Random displacement within a fixed maximum distance, optionally differentiated
  by urban and rural areas.

- **Donut geomasking**  
  Enforces a minimum displacement distance to reduce the risk of re-identification
  near the original location.

- **Population-adaptive displacement**  
  Dynamically adjusts the displacement radius to ensure that a minimum population
  threshold surrounds the anonymized point.

---

## 2. Anonymization Logic in This Package

### 2.1 The Three-Stage Anonymization Model

The library follows a clear three-stage mental model that separates *privacy policy*
from *geometry* and *validity checks*:

1. **Radius Selection (Privacy Envelope)**  
   Determines how large the uncertainty region should be based on the chosen method
   (fixed DHS radius, donut bounds, or population thresholds).

2. **Point Displacement (Geometry)**  
   Generates a candidate displaced point using a random bearing and a sampled distance
   within the selected radius.

3. **Constraint Evaluation (Validity)**  
   Verifies whether the displaced point satisfies spatial constraints such as
   administrative boundaries, water-body exclusion, or road buffers.

**Key principle:**  
*Population determines the anonymization envelope; constraints determine whether a
candidate point is acceptable.*

This separation:
- prevents infinite retry loops,
- ensures predictable runtime,
- and makes each stage independently testable and auditable.

---

## 3. Dataset Description

### 3.1 Dataset Overview

This notebook uses real survey data from a household survey conducted in **Malawi**.
The dataset contains GPS coordinates and household metadata collected during
field operations.

**Source:** Malawi Household Survey  
**Geographic Coverage:** Multiple districts across Malawi  
**Spatial Units:** Individual households within enumeration areas (EAs)

---

### 3.2 Key Columns

#### Identifiers and Geography
- `HHID`: Unique household identifier  
- `ea_id`: Enumeration area identifier  
- `ea_code`: Enumeration area code  
- `district`: Administrative district name  

#### Survey Metadata
- `interviewDate`: Date when household was interviewed  
- `reside`: Residence classification (urban/rural)  
- `camp`: Settlement or camp indicator  
- `hh_a02a`, `hh_a02b`: Additional household identifiers  

---

### 3.3 GPS Data Sources

#### Garmin GPS Device
- `latitude_garmin`: Latitude from Garmin GPS (primary coordinates)  
- `longitude_garmin`: Longitude from Garmin GPS (primary coordinates)  

#### Tablet GPS
- `latitude_tablet`: Latitude from tablet GPS  
- `longitude_tablet`: Longitude from tablet GPS  
- `accuracy_tablet`: GPS accuracy in meters  
- `altitude_tablet`: Elevation above sea level  
- `timestamp_tablet`: GPS timestamp  
- `weather_tablet`: Weather conditions during GPS capture  
- `satellites_tablet`: Number of satellites used for position fix  

---

### 3.4 Data Characteristics

- **Dual GPS sources** enable cross-checking and quality control  
- **Quality metrics** (accuracy, satellites, weather) support reliability assessment  
- **Geographic hierarchy**: district → enumeration area → household  
- **Urban/rural classification** allows differentiated anonymization policies  

---

### 3.5 Privacy Considerations

The raw GPS coordinates in this dataset are **highly confidential** and must be
anonymized before dissemination in order to:

- prevent household re-identification,
- protect respondent privacy,
- comply with data protection and ethical research standards.

This notebook demonstrates how to anonymize these coordinates using multiple
methods while preserving geographic utility for downstream analysis.

---

## 4. Notebook Structure

The remainder of this notebook is organized as follows:

1. [Python Environment Setup](#setup)  
2. [Working Directory and Data Layout](#working-directory)  
3. [Simple API: Basic Usage](#simple-basic)  
4. [Simple API: Automatic Data Downloads](#simple-download)  
5. [Simple API: Applying Constraints](#simple-constraints)  
6. [Config-Based API: Full Control](#config-based)  
7. [Config-Based API: Loading from YAML](#config-yaml)  
8. [Advanced Usage: User-Provided Spatial Data](#advanced-user-data)  


## 1. Python Setup <a id='setup'></a>

1. Ensure that you have the package installed by following instructions provided on the GitHub project documentation.
2. Import required libraries 

In [1]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

from spatial_anonymization.api import (
    anonymize_survey_coordinates, 
    validate_input_schema
    )

from spatial_anonymization.config.params import (
    AnonymizationParams,
    DataSourcesConfig,
)

# Import configuration presets from config_presets module
from spatial_anonymization.config.presets import (
    DHS,
    WB_LSMS,
    get_preset,
    list_presets,
)


## 2. Working Directory and Inputs Setup

In this section, define global path variables that point to your working directory and
the main data folders used throughout the anonymization workflow.

These paths will be used to:
- load existing input datasets (e.g. survey coordinates, admin boundaries)
- download and cache external resources (e.g. population rasters, water bodies)
- save anonymized outputs, diagnostics, and configuration files

It is recommended to centralize all file paths at the beginning of your script or notebook
to ensure clarity, portability, and reproducibility. For consistency, you start folders with ```DIR``` and files with ```FILE```. 

### Example

```python
from pathlib import Path

# Base working directory
DIR_WORK = Path("/path/to/your/project")

# Input data directories
DIR_DATA = DIR_WORK / "data"
DIR_RAW_DATA = DATA_DIR / "raw"
DIR_PROCESSED_DATA = DATA_DIR / "processed"

In [ ]:
# ======================================
# SETUP INPUT PATHS CONSTANTS
# ======================================
# Base working directory
DIR_WORK = Path("/Users/dmatekenya/Library/CloudStorage/OneDrive-WBG/Survey-Geo-Enhancement/git-repos/spatial-anonymization/")

# Input data directories
DIR_DATA = DIR_WORK / "data"

# Malawi Test Data
DIR_MWI = DIR_DATA / "mwi"
DIR_MWI_GEOSPATIAL = DIR_MWI / "geospatial"
FILE_MWI_CENSUS_EAS = DIR_MWI_GEOSPATIAL / "mwi-2018-census-eas-wg84.shp"
FILE_MWI_CENSUS = DIR_MWI / "census/BT-census-hhs-50k-sample.csv"
FILE_MWI_IHS6 = DIR_MWI / "ihs6-2024-25/cleaned-test.csv"


# ========================
# INPUT FILES FOR MALAWI
# ========================
# DIR_ETH_SURVEY = DIR_DATA.joinpath('eth', 'survey')
# ETH_GEOVARS = DIR_DATA.joinpath('eth', 'w4_geovars')

# # GPS coordinates file
# # ETH_COORDS = ETH_SURVEY.joinpath("HH level coordinates.dta")
# FILE_ETH_COORDS = DIR_ETH_SURVEY.joinpath("ETH_gps_work_r12082021.csv")

# # Admin regions (adm2) shapefile
# ETH_ADM = ETH_GEOVARS.joinpath("spatial_data", "shp_adm2", "eth_admbnda_adm2_csa_bofed_20190827.shp")

# # output directory
# ETH_OUPUT_DIR = DATA.joinpath("ETH", "outputs")

## 3. Coordinates Data Preparation

### 3.1 Review and Preprocess Coordinates

This package assumes that users have already performed the necessary data cleaning and preprocessing on their survey GPS coordinates prior to anonymization.

At a minimum, the input dataset is expected to:

- Contain longitude and latitude columns in decimal degrees (WGS84)
- Have numeric coordinate values
- Have no missing (null) values in the coordinate columns


### 3.2 pecifiy Input Data Schema 

The package needs to understand how geographic and contextual information is represented in your dataset. In this step, users explicitly specify which columns contain longitude and latitude values, as well as any auxiliary fields used to guide anonymization policies.

In particular, users need to indicate:
- The column names containing **longitude** and **latitude** values
- The column used for **urban/rural classification**, if available
- Which value or label corresponds to **urban** versus **rural** areas


In [3]:
# ============================================================
# SPECIFY INPUT DATA SCHEMA
# ============================================================
lat = "latitude_tablet"
lon = "longitude_tablet"
urban_rural = "urb_rur"
urban_value = "URBAN"
rural_value = "RURAL"

In [4]:
# ======================================
# VALIDATE INPUT DATA
# =======================================
# Load the data
df_mwi_ihs = pd.read_csv(FILE_MWI_IHS6)

# Check the data
validate_input_schema(df_mwi_ihs, lat_col=lat, lon_col=lon,
                      urban_rural_col=urban_rural, urban_value="URBAN",
                      rural_value="RURAL")


All Input Data Schema Validations Passed!


## Configuring Spatial Anonymization

Before running the anonymization process, users need to make a small number of **explicit decisions** about *how* spatial privacy should be enforced. These decisions are expressed through **anonymization parameters**, which control the displacement method, distance thresholds, and optional constraints.

The package supports two complementary ways of specifying these choices:

### 1. Use a Preset Configuration (Recommended)
For common use cases—such as DHS-style household or cluster anonymization—the package provides **predefined parameter presets**. These presets bundle together well-established methodological choices (e.g., urban vs. rural distances, population thresholds) and are a good starting point for most users.

Presets allow users to:
- Adopt standard methodologies with minimal setup
- Ensure consistency with published practices (e.g., DHS)
- Override only a small number of parameters when needed

### 2. Define Custom Parameters
For advanced or non-standard use cases, users can construct their own parameter configuration by explicitly setting anonymization options. This provides full control over:
- Displacement method (e.g., DHS, donut, population-adaptive)
- Maximum and minimum displacement distances
- Urban/rural–specific rules
- Population thresholds
- Which spatial constraints are enforced

### Key Decisions Users Should Make
At a minimum, users should decide:
- **Which anonymization method** to use
- **Whether distances differ by urban/rural classification**
- **Whether population thresholds should be enforced**
- **Which spatial constraints are required** (e.g., admin boundaries, water bodies)

These choices reflect both **privacy requirements** and **analytical needs**. The package is designed so that all such decisions are explicit, transparent, and reproducible—either through presets or user-defined parameters.


In [7]:
df_mwi_stata = pd.read_stata(FILE_STATA_IHS6)

In [8]:
# Columns to Keep
cols = ['HHID', 'reside', 
       'ea_code', 'latitude_tablet',
       'longitude_tablet']
df_mwi_stata = df_mwi_stata[cols]

# Rename columns
df_mwi_stata.rename(columns={'HHID': "hh_id", 'reside': "urb_rur"}, inplace=True)

# Ensure that urban rural has only 2 values 
assert df_mwi_stata.urb_rur.value_counts().shape[0] == 2

# Ensure all lat/lon values are numeric and not null
# Drop null values 
df_mwi_stata = df_mwi_stata.dropna(subset=['latitude_tablet', 'longitude_tablet'])


In [93]:
df_mwi_stata.to_csv(DIR_MWI/ "ihs6-2024-25/cleaned-test.csv", index=False)

## 4. Configuring Anonymization and Preparing Geospatial Inputs
In utilizing this package, there different decisions on methodology and specific configurations and threshold you have to set. Here we provide the key ones.

- Anpymization approach. DHS, Donut, Population adapative or a combination of the these
- Inpu geospatial layers. Decide on source of admin boundaries, population, water bodies with the two options being either you provide your own layer or let the package downloa
- Input data schema. Indicate which columns contains lat/long, urban/rural cetegprzation.
- Distance and popul;ation thresholds. Distance thresholds for displacement from original point


In [10]:
df_mwi_stata.columns

Index(['hh_id', 'urb_rur', 'ea_code', 'latitude_tablet', 'longitude_tablet'], dtype='object')

## 5. Test Functions for Different Anonymization Scenarios

Below are reusable test functions that demonstrate various anonymization scenarios.
These functions can be used both in this notebook and in automated tests.

Each function:
- Takes a sample of the Malawi IHS6 data
- Applies a specific anonymization configuration
- Returns the anonymized dataframe
- Includes validation checks to ensure successful anonymization

## 2. Simple API - Basic Usage <a id='simple-basic'></a>

The simplest way to anonymize coordinates using DHS-style displacement with auto-downloaded admin boundaries.

In [ ]:
# Basic DHS-style anonymization with auto-download
df_anon_basic = anonymize_survey_data(
    df=sample_data,
    country_iso3="MWI",        # Malawi
    admin_level=2,              # District level
    lon_col="lon",
    lat_col="lat",
    method="dhs",
    max_distance=2000,          # 2km displacement
    working_dir=str(WORKING_DIR),
    random_seed=42,
)

print("\nAnonymized data (basic):")
df_anon_basic[['household_id', 'lat', 'lon', 'lat_anon', 'lon_anon']].head()

## 3. Simple API - Auto-Download Data <a id='simple-download'></a>

Automatically download population and water data for population-adaptive anonymization.

In [ ]:
# Population-adaptive with auto-downloaded population raster
df_anon_download = anonymize_survey_data(
    df=sample_data,
    country_iso3="MWI",
    admin_level=2,
    lon_col="lon",
    lat_col="lat",
    method="population_adaptive",
    auto_download_population=True,
    population_year=2025,
    working_dir=str(WORKING_DIR),
    random_seed=42,
)

print("\nAnonymized data (with auto-download):")
df_anon_download[['household_id', 'lat', 'lon', 'lat_anon', 'lon_anon']].head()

## 4. Simple API - With Constraints <a id='simple-constraints'></a>

Apply spatial constraints to avoid water bodies and respect admin boundaries. Supports urban/rural differentiation.

In [ ]:
# DHS with urban/rural differentiation and water constraints
df_anon_constraints = anonymize_survey_data(
    df=sample_data,
    country_iso3="MWI",
    admin_level=2,
    lon_col="lon",
    lat_col="lat",
    method="dhs",
    # Urban/rural settings
    urban_col="urban_rural",
    urban_value="urban",
    rural_value="rural",
    urban_max_distance=2000,    # 2km for urban
    rural_max_distance=5000,    # 5km for rural
    # Constraints
    enforce_admin_boundary=True,
    enforce_water_constraint=True,
    auto_download_water=True,
    working_dir=str(WORKING_DIR),
    random_seed=42,
)

print("\nAnonymized data (with constraints):")
df_anon_constraints[['household_id', 'urban_rural', 'lat', 'lon', 'lat_anon', 'lon_anon']].head()

## 4.5 Using Config Presets - Quick Start with Standard Configurations

The library provides predefined configuration presets for common survey workflows (DHS, World Bank LSMS). 
Presets are pre-configured `AnonymizationConfig` objects that follow established survey standards, 
eliminating the need to manually configure 15+ parameters while still allowing customization.

**Understanding the Relationship:**
- `AnonymizationConfig` = Empty configuration form (you fill all fields)
- `Preset` = Pre-filled configuration following standards (DHS, LSMS) that you can customize

### Available Presets

1. **DHS** - Standard DHS displacement methodology
   - Urban: 2km max displacement
   - Rural: 5km max displacement  
   - Rural subset (1%): 10km displacement
   
2. **WB_LSMS** - World Bank LSMS standards
   - Similar displacement to DHS (10% rural extra instead of 1%)
   - Includes population threshold (500 people minimum)
   - Suitable for LSMS-ISA, IHS, and other WB surveys

In [ ]:
# Import preset utilities from config_presets module
from spatial_anonymization.config.presets import (
    DHS, 
    WB_LSMS, 
    get_preset, 
    list_presets
)

# Display all available presets
list_presets()

### Example 1: Use DHS Preset Directly

The simplest approach - use a preset with minimal configuration.

In [ ]:
# Use DHS preset directly - just specify your data columns
df_anon_dhs = anonymize_survey_data_from_config(
    df=df_mwi_stata.head(100),  # Using sample of prepared Malawi data
    config=DHS,
    country_iso3="MWI",
    random_seed=42,
)

print("Anonymized using DHS preset:")
print(f"  Input rows: {len(df_mwi_stata.head(100))}")
print(f"  Output rows: {len(df_anon_dhs)}")
print(f"\nFirst few results:")
df_anon_dhs[['hh_id', 'urb_rur', 'latitude_tablet', 'longitude_tablet', 
             'lat_anon', 'lon_anon']].head()

### Example 2: Customize a Preset

Start with a preset and modify only the parameters you need to change. This is more convenient than configuring everything from scratch.

In [ ]:
from dataclasses import replace

# Start with WB_LSMS preset and customize for your data
custom_config = replace(
    WB_LSMS,
    # Override column names to match your data
    lat_col="latitude_tablet",
    lon_col="longitude_tablet", 
    urban_rural_col="urb_rur",
    # Increase urban displacement to 3km
    urban_max_distance=3000,
    # Change unit type to household
    unit_type="household"
)

print("Custom configuration based on WB_LSMS:")
print(f"  Method: {custom_config.method}")
print(f"  Unit type: {custom_config.unit_type}")
print(f"  Urban max distance: {custom_config.urban_max_distance}m")
print(f"  Rural max distance: {custom_config.rural_max_distance}m")
print(f"  Lat column: {custom_config.lat_col}")
print(f"  Lon column: {custom_config.lon_col}")
print(f"  Min population: {custom_config.min_population}")

### Example 3: Get Preset by Name

Useful when the preset name comes from a configuration file or user input.

In [ ]:
# Get preset by name (case-insensitive)
config_dhs = get_preset("dhs")
config_lsms = get_preset("wb_lsms")  # or "lsms" works too

print("Retrieved DHS preset:")
print(f"  Urban max: {config_dhs.urban_max_distance}m")
print(f"  Rural max: {config_dhs.rural_max_distance}m")
print(f"  Rural extra proportion: {config_dhs.rural_extra_proportion * 100}%")

print("\nRetrieved WB_LSMS preset:")
print(f"  Urban max: {config_lsms.urban_max_distance}m")
print(f"  Rural max: {config_lsms.rural_max_distance}m")
print(f"  Rural extra proportion: {config_lsms.rural_extra_proportion * 100}%")
print(f"  Min population: {config_lsms.min_population}")

# Can also customize after retrieval
custom = replace(get_preset("dhs"), lat_col="latitude_garmin", lon_col="longitude_garmin")
print(f"\nCustomized DHS preset - Lat column: {custom.lat_col}, Lon column: {custom.lon_col}")

### Benefits of Using Config Presets

**With Config Presets:**
- ✅ Start with proven configurations (DHS, LSMS standards)
- ✅ Only override what you need to change
- ✅ Ensures consistency across projects
- ✅ Self-documenting (preset name indicates methodology)
- ✅ Easy to discover available options with `list_presets()`
- ✅ Clear import structure: `from spatial_anonymization.config_presets import DHS`

**Without Presets:**
- ⚠️ Must configure 15+ parameters manually
- ⚠️ Risk of inconsistent configurations
- ⚠️ Harder to ensure compliance with standards

**When to use each preset:**
- **DHS**: Use for health/demographic surveys following DHS methodology
- **WB_LSMS**: Use for World Bank LSMS surveys or when population thresholds are required

**Import Pattern:**
```python
# Import presets from config_presets module (clear and explicit)
from spatial_anonymization.config_presets import DHS, WB_LSMS, get_preset, list_presets

# Core config class if building from scratch
from spatial_anonymization import AnonymizationConfig
```

## 5. Config-Based API - Full Control <a id='config-based'></a>

Use configuration objects for maximum control and reproducibility.

In [ ]:
# Create custom data sources configuration
data_sources = DataSourcesConfig(
    admin_boundary_source="geoboundaries",
    admin_boundary_fallback="gadm",
    population_source="worldpop",
    population_dataset_version="R2025A_v1",
    population_year=2025,
    population_resolution="100m",
    population_type="constrained",
    water_source="osm",
    working_dir=str(WORKING_DIR),
    cache_downloads=True,
    auto_download=True,
)

# Create anonymization configuration
config = AnonymizationConfig(
    method="donut",
    unit_type="household",
    # Distance policy
    min_distance=500,           # Minimum 500m displacement
    max_distance=2000,          # Maximum 2km displacement
    urban_min_distance=300,
    urban_max_distance=1500,
    rural_min_distance=500,
    rural_max_distance=5000,
    # Input schema
    lat_col="lat",
    lon_col="lon",
    urban_rural_col="urban_rural",
    urban_value="urban",
    rural_value="rural",
    # Constraints
    admin_level=2,
    enforce_admin_boundary=True,
    enforce_water_constraint=True,
    water_constraint_strict=True,
    # Data sources
    data_sources=data_sources,
)

print("Configuration created:")
print(f"  Method: {config.method}")
print(f"  Min distance: {config.min_distance}m")
print(f"  Max distance: {config.max_distance}m")
print(f"  Population source: {config.data_sources.population_source}")
print(f"  Population year: {config.data_sources.population_year}")

In [ ]:
# Use the config-based API
df_anon_config = anonymize_survey_data_from_config(
    df=sample_data,
    config=config,
    country_iso3="MWI",
    random_seed=42,
)

print("\nAnonymized data (config-based):")
df_anon_config[['household_id', 'urban_rural', 'lat', 'lon', 'lat_anon', 'lon_anon']].head()

## 6. Config-Based API - From YAML <a id='config-yaml'></a>

Save and load configurations from YAML files for reproducibility.

In [ ]:
# Save configuration to YAML
config_path = WORKING_DIR / "anonymization_config.yaml"
config.to_yaml(config_path)

print(f"Configuration saved to: {config_path}")
print("\nYAML content preview:")
print("-" * 60)
with open(config_path, 'r') as f:
    content = f.read()
    # Print first 30 lines
    lines = content.split('\n')[:30]
    print('\n'.join(lines))
    if len(content.split('\n')) > 30:
        print("\n... (truncated)")

In [ ]:
# Note: Loading from YAML requires implementing from_yaml() method
# For now, you can manually edit the YAML and reconstruct the config

# Example of what loading would look like (to be implemented):
# config_loaded = AnonymizationConfig.from_yaml(config_path)
# df_anon = anonymize_survey_data_from_config(
#     df=sample_data,
#     config=config_loaded,
#     country_iso3="MWI"
# )

print("To load config from YAML, implement AnonymizationConfig.from_yaml() method")

## 7. Advanced: User-Provided Data <a id='advanced-user-data'></a>

Use your own shapefiles and rasters instead of auto-downloading.

In [ ]:
# Example with user-provided admin boundaries
# Assuming you have a shapefile at this path:
# admin_shapefile = "../data/MW-test/Malawi_TA_2018.shp"

# If file exists, use it:
admin_file = Path("../data/MW-test/Malawi_TA_2018.shp")

if admin_file.exists():
    print(f"Using existing admin boundaries: {admin_file}")
    
    df_anon_user = anonymize_survey_data(
        df=sample_data,
        admin_boundaries=str(admin_file),  # User-provided shapefile
        lon_col="lon",
        lat_col="lat",
        method="dhs",
        max_distance=2000,
        enforce_admin_boundary=True,
        enforce_water_constraint=False,  # No water constraints
        working_dir=str(WORKING_DIR),
        random_seed=42,
    )
    
    print("\nAnonymized data (user-provided boundaries):")
    display(df_anon_user[['household_id', 'lat', 'lon', 'lat_anon', 'lon_anon']].head())
else:
    print(f"Admin shapefile not found at {admin_file}")
    print("Using auto-download instead...")
    
    df_anon_user = anonymize_survey_data(
        df=sample_data,
        country_iso3="MWI",
        admin_level=3,  # Use Traditional Authority level
        lon_col="lon",
        lat_col="lat",
        method="dhs",
        max_distance=2000,
        working_dir=str(WORKING_DIR),
        random_seed=42,
    )
    
    print("\nAnonymized data (auto-downloaded):")
    display(df_anon_user[['household_id', 'lat', 'lon', 'lat_anon', 'lon_anon']].head())

## Summary

This notebook demonstrated:

1. **Simple API (`anonymize_survey_data`)**:
   - Basic DHS-style displacement
   - Auto-download of admin boundaries and population data
   - Urban/rural differentiation
   - Spatial constraints (admin boundaries, water bodies)

2. **Config-Based API (`anonymize_survey_data_from_config`)**:
   - Full control via `AnonymizationConfig` and `DataSourcesConfig`
   - Save/load configurations for reproducibility
   - Custom data source providers

3. **Advanced Features**:
   - User-provided shapefiles and rasters
   - Donut geomasking with minimum displacement
   - Population-adaptive methods

### Next Steps

- Experiment with different methods (`dhs`, `donut`, `population_adaptive`)
- Try different distance policies for urban vs rural areas
- Visualize original vs anonymized coordinates on a map
- Evaluate anonymization quality with diagnostic metrics